In [1]:
using Oscar
using HomotopyContinuation
using LinearAlgebra

V = [[4,1], [5,3], [2,4], [0,2], [1,1]]
#V=[[0,0], [0,1], [1,0]]
P = convex_hull(V)

A = volume(P) 
println(A)

  ___   ___   ___    _    ____
 / _ \ / __\ / __\  / \  |  _ \  | Combining and extending ANTIC, GAP,
| |_| |\__ \| |__  / ^ \ |  ´ /  | Polymake and Singular
 \___/ \___/ \___//_/ \_\|_|\_\  | Type "?Oscar" for more information
o--------o-----o-----o--------o  | Documentation: https://docs.oscar-system.org
  S Y M B O L I C   T O O L S    | Version 1.7.2
10


In [2]:
a = 5
b = 10
c = -40
pos = Int[]
for (i,v) in enumerate(V)
    s = a*v[1]+b*v[2] + c
    push!(pos, sign(s))
end
println(pos)

[-1, 1, 1, -1, -1]


In [3]:
cut = Vector{Vector{Int}}()
edgespm = Polymake.to_one_based_indexing(Polymake.polytope.faces_of_dim(P.pm_polytope, 1))
edges = [collect(e) for e in edgespm]
    for e in edges
        v1 = vertices(P)[e[1]]
        v2 = vertices(P)[e[2]]
    
        s1 = a*v1[1]+b*v1[2] +c
        s2 = a*v2[1]+b*v2[2] +c
        if s1*s2 < 0
            push!(cut, e)
        end
    end
println(cut)
println(cut[1])
println(cut[1][1])
println(V[cut[1][1]])

[[3, 4], [1, 2]]
[3, 4]
3
[2, 4]


In [4]:
newpoints = Vector{Vector{QQFieldElem}}()
println(length(cut))
for (i,v) in enumerate(cut)
    v1 = V[v[1]]
    v2 = V[v[2]]
    l = (a * v2[1] + b * v2[2] + c) // (a * v2[1] + b * v2[2] - a * v1[1] - b * v1[2])
    p = l*v1 + (1-l)*v2
    push!(newpoints, p)
    println(p)
end
println(newpoints)

2
Rational{Int64}[4//3, 10//3]
Rational{Int64}[22//5, 9//5]
Vector{QQFieldElem}[[4//3, 10//3], [22//5, 9//5]]


In [5]:
points = Vector{Vector{QQFieldElem}}()
for (i,v) in enumerate(V)
    s = a*v[1]+b*v[2] + c
    if s > 1
        push!(points, v)        
    end
end
for (i,v) in enumerate(newpoints)
    push!(points, v)
end    
Pplus = convex_hull(QQ, points)
println( volume(Pplus))


109//30


In [6]:
embedding = Vector{Vector{QQFieldElem}}()
for (i, v) in enumerate(V)
    v_lift = [v[1], v[2], QQ(1)]
    push!(embedding,v_lift)
end
println(embedding)

Vector{QQFieldElem}[[4, 1, 1], [5, 3, 1], [2, 4, 1], [0, 2, 1], [1, 1, 1]]


In [7]:
arrangement = Vector{Polyhedron{QQFieldElem}}()
for (i, v) in enumerate(embedding)
    #v_lift = [v[1], v[2], QQ(1)]
    #println(v_lift)    
    # clear denominators to get integer vector
    l = lcm([denominator(c) for c in v])
    v_int = [Int(numerator(l * c)) for c in v]
    vi_perp = polyhedron(transpose([v_int -v_int]), [0; 0])

    println("Lineality space: ", lineality_space(vi_perp))
    push!(arrangement, vi_perp)
    end
println(arrangement)

Lineality space: RayVector{QQFieldElem}[[-1//4, 1, 0], [-4//17, -1//17, 1]]
Lineality space: RayVector{QQFieldElem}[[-3//5, 1, 0], [-5//34, -3//34, 1]]
Lineality space: RayVector{QQFieldElem}[[-2, 1, 0], [-1//10, -1//5, 1]]
Lineality space: RayVector{QQFieldElem}[[1, 0, 0], [0, -1//2, 1]]
Lineality space: RayVector{QQFieldElem}[[-1, 1, 0], [-1//2, -1//2, 1]]
Polyhedron{QQFieldElem}[Polyhedron in ambient dimension 3, Polyhedron in ambient dimension 3, Polyhedron in ambient dimension 3, Polyhedron in ambient dimension 3, Polyhedron in ambient dimension 3]


In [8]:
M = matrix(QQ, length(embedding),3, vcat([[v[1], v[2], v[3]] for v in embedding]...))

[4   1   1]
[5   3   1]
[2   4   1]
[0   2   1]
[1   1   1]

In [9]:
HA = Polymake.fan.HyperplaneArrangement(HYPERPLANES=M)
CD = HA.CHAMBER_DECOMPOSITION

fan = polyhedral_fan(CD)
chambers = maximal_cones(fan)

println(chambers)
println(n_maximal_cones(fan))

Cone{QQFieldElem}[Polyhedral cone in ambient dimension 3, Polyhedral cone in ambient dimension 3, Polyhedral cone in ambient dimension 3, Polyhedral cone in ambient dimension 3, Polyhedral cone in ambient dimension 3, Polyhedral cone in ambient dimension 3, Polyhedral cone in ambient dimension 3, Polyhedral cone in ambient dimension 3, Polyhedral cone in ambient dimension 3, Polyhedral cone in ambient dimension 3, Polyhedral cone in ambient dimension 3, Polyhedral cone in ambient dimension 3, Polyhedral cone in ambient dimension 3, Polyhedral cone in ambient dimension 3, Polyhedral cone in ambient dimension 3, Polyhedral cone in ambient dimension 3, Polyhedral cone in ambient dimension 3, Polyhedral cone in ambient dimension 3, Polyhedral cone in ambient dimension 3, Polyhedral cone in ambient dimension 3, Polyhedral cone in ambient dimension 3, Polyhedral cone in ambient dimension 3]
22


In [10]:
function hyperplane_intersectedge(V,a,b,c)
	P = convex_hull(QQ,V)
	cut = []
	for e in faces(P,1)
		s = [a*v[1]+b*v[2] + c for v in vertices(e)]
		if prod(s) < 0
           		push!(cut, e)
        		end
	end
	return cut #Chiara: this now outputs two edges instead of just the indices of the corresponding vertices
end

hyperplane_intersectedge (generic function with 1 method)

In [11]:
#PRE: list of points vertices that spans the polytope and let a,b,c be the coefficients that describe the hyperplane H={ax+by+c=0}
#POST: returns a vector containing the exact newpoints where H cuts the polygon spanend by P
function hyperplane_intersectpoints(V,a,b,c)
    edges = hyperplane_intersectedge(V,a,b,c)
    newpoints = Vector{Vector{QQFieldElem}}()
    for e in edges
        v = vertices(e)
        s1 = inner_product(v[2],a,b,c)
        s2 = inner_product(v[1],a,b,c)
        l = s1 //(s1 - s2)
        p = [l*v[1][i] + (1-l)*v[2][i] for i in 1:2]
        push!(newpoints, p)
    end
return newpoints
end

hyperplane_intersectpoints (generic function with 1 method)

In [12]:
for ch in chambers
    println(sum(rays(ch)))
    println(facets(ch))
end

QQFieldElem[0, -3//2, 33//2]
5-element SubObjectIterator{LinearHalfspace{QQFieldElem}} over the halfspaces of R^3 described by:
-5*x_1 - 3*x_2 - x_3 <= 0
-4*x_1 - x_2 - x_3 <= 0
-2*x_1 - 4*x_2 - x_3 <= 0
-2*x_2 - x_3 <= 0
-x_1 - x_2 - x_3 <= 0

QQFieldElem[3, -1//3, -2//3]
3-element SubObjectIterator{LinearHalfspace{QQFieldElem}} over the halfspaces of R^3 described by:
2*x_2 + x_3 <= 0
-2*x_1 - 4*x_2 - x_3 <= 0
-x_1 - x_2 - x_3 <= 0

QQFieldElem[1, -9, 26]
3-element SubObjectIterator{LinearHalfspace{QQFieldElem}} over the halfspaces of R^3 described by:
-5*x_1 - 3*x_2 - x_3 <= 0
2*x_1 + 4*x_2 + x_3 <= 0
-2*x_2 - x_3 <= 0

QQFieldElem[4, -25//3, 37//3]
4-element SubObjectIterator{LinearHalfspace{QQFieldElem}} over the halfspaces of R^3 described by:
2*x_2 + x_3 <= 0
-5*x_1 - 3*x_2 - x_3 <= 0
-x_1 - x_2 - x_3 <= 0
2*x_1 + 4*x_2 + x_3 <= 0

QQFieldElem[2, 6, -11]
3-element SubObjectIterator{LinearHalfspace{QQFieldElem}} over the halfspaces of R^3 described by:
x_1 + x_2 + x_3 <= 0
-4*x_1

In [13]:
samplevectors = []
    for ch in chambers
        u = sum(rays(ch))
        edgecut = hyperplane_intersectedge(V, u[1],u[2],u[3])
        if length(edgecut) == 2
            push!(samplevectors, (chamber = ch, repr = collect(QQFieldElem, u)))
        end
    end
    return samplevectors

20-element Vector{Any}:
 (chamber = Polyhedral cone in ambient dimension 3, repr = QQFieldElem[3, -1//3, -2//3])
 (chamber = Polyhedral cone in ambient dimension 3, repr = QQFieldElem[1, -9, 26])
 (chamber = Polyhedral cone in ambient dimension 3, repr = QQFieldElem[4, -25//3, 37//3])
 (chamber = Polyhedral cone in ambient dimension 3, repr = QQFieldElem[2, 6, -11])
 (chamber = Polyhedral cone in ambient dimension 3, repr = QQFieldElem[4, 16//3, -46//3])
 (chamber = Polyhedral cone in ambient dimension 3, repr = QQFieldElem[-2, 7//2, 3//2])
 (chamber = Polyhedral cone in ambient dimension 3, repr = QQFieldElem[-1, 12, -20])
 (chamber = Polyhedral cone in ambient dimension 3, repr = QQFieldElem[-3, -19//6, 133//6])
 (chamber = Polyhedral cone in ambient dimension 3, repr = QQFieldElem[-4, 13//6, 47//6])
 (chamber = Polyhedral cone in ambient dimension 3, repr = QQFieldElem[-2, -38//3, 110//3])
 (chamber = Polyhedral cone in ambient dimension 3, repr = QQFieldElem[-4, -16//3, 46//3])
 (c

In [14]:
#PRE: list of vertices that span the polytope, field S and coefficients of the hyperplane a,b,c
#POST: returns the condition of a vertex to be on the same side wrt to the hyperplane ax+by+c=0
function points_above(V,S,a,b,c)
    points = Vector{Vector{eltype(S)}}()
        #using the representing vector, understand which point lie above the hyperplane, so that we can calculate the area cut
    #VL = [[S(v[1]), S(v[2])] for v in V]    
    for v in V
            s = inner_product(v,a,b,c)
            if s > 0
                #push!(points, VL[i])
                push!(points, [S(v[1]), S(v[2])])
            end
        end
    return points
end


points_above (generic function with 1 method)

In [15]:
#PRE: a point v and coefficients a,b,c defining an hyperplane ax+by+c=0
#POST: returns the dot product (a b c). (v 1)
function inner_product(v,a,b,c)
    v_lift = [v[1], v[2], QQ(1)]
    s = dot([a,b,c],v_lift)
    return s
end
    

inner_product (generic function with 1 method)

In [16]:
g = vertex_edge_graph(P)

start = 1
prev  = start
curr  = neighbors(g, start)[1] # take one neighbor of the starting vertex "start"
ord   = [start, curr]

while true
    ns  = neighbors(g, curr)
    # take the next neighbor, not the one we came from
    if ns[1] == prev
        nxt = ns[2]
    else
        nxt = ns[1]
    end
    # check if the next neighbor is your starting point: in that case you are done!
    if nxt == start
        break
    end

    push!(ord, nxt)
    prev, curr = curr, nxt # update and keep going
end
println(ord)

[1, 2, 3, 4, 5]


In [17]:
##TO IMPROVE, IS NOT OPTIMAL, WORKS ONLY IF VERTICES ARE IN A CYCLIC ORDER##
#PRE: list of vertices that span the polytope 
#POST: returns the area of the spanned polytope without needing to create its convex hull
function shoelace(V)
    n = length(V)
    area = sum(V[i][1]*V[mod(i,n)+1][2] - 
               V[mod(i,n)+1][1]*V[i][2] for i in 1:n)
    return area // 2
end


shoelace (generic function with 1 method)

In [18]:
#PRE: V are the vertices of a polytope
#POST: returns the the vertices ordered. This might be clockwise or not.
function cyclic_order(V) 
    P = convex_hull(V)
    g = vertex_edge_graph(P)

    start = 1
    prev  = start
    curr  = neighbors(g, start)[1] # take one neighbor of the starting vertex "start"
    ord   = [start, curr]

    while true
        ns  = neighbors(g, curr)
        # take the next neighbor, not the one we came from
        if ns[1] == prev
            nxt = ns[2]
        else
            nxt = ns[1]
        end
        # check if the next neighbor is your starting point: in that case you are done!
        if nxt == start
            break
        end

        push!(ord, nxt)
        prev, curr = curr, nxt # update and keep going
    end

    return ord
end

cyclic_order (generic function with 1 method)

In [19]:
R, (a, b, c) = polynomial_ring(QQ, ["a", "b", "c"])
S = fraction_field(R)

fcts = []
VL = [[S(v[1]), S(v[2])] for v in V]

for (r ,u) in samplevectors
        edgecut = hyperplane_intersectedge(V, u[1], u[2], u[3])
        
        newpoints = Vector{Vector{eltype(S)}}()
        newpoints_eval = Vector{Vector{QQFieldElem}}()
        for e in edgecut
            v = [S.(collect(vv)) for vv in vertices(e)]
            v_eval = [collect(vv) for vv in vertices(e)]
            l = (dot([a, b], v[2]) + c) //(dot([a, b], v[2]) - dot([a, b], v[1]))
            l_eval = inner_product(v_eval[2],u[1], u[2], u[3]) //(inner_product(v_eval[2],u[1], u[2], u[3]) - inner_product(v_eval[1],u[1], u[2], u[3]))
            p = [l*v[1][i] + (1-l)*v[2][i] for i in 1:2]
            p_eval = [l_eval*v_eval[1][i] + (1-l_eval)*v_eval[2][i] for i in 1:2]
            push!(newpoints, p)
            push!(newpoints_eval, p_eval)
        end

        # pts = points_above(V, S, u[1], u[2], u[3])
        push!(newpoints, points_above(V, S, u[1], u[2], u[3])...)
        push!(newpoints_eval, points_above(V, QQ, u[1], u[2], u[3])...)
    
        # now we order the points cyclically, using the evaluated vertices, and we check the sign of the shoelace function
        order = cyclic_order(newpoints_eval)
        sgn = sign(shoelace(newpoints_eval[order]))
    
        local_vol = sgn*shoelace(newpoints[order])
        push!(fcts, (chamber = r, repr = u, local_area = local_vol))
end

f = 2*a+3*b*c+5//2*a*b^2

result = HomotopyContinuation.Expression(0)
@var a b c
vars = [a,b,c]
for t in terms(f)
    c = collect(AbstractAlgebra.coefficients(t))
    e = collect(AbstractAlgebra.exponent_vectors(t))
    monom = prod(vars[i]^e[1][i] for i in 1:3)
    coeff = numerator(c[1])//denominator(c[1])
    term = Rational(coeff)*monom
    result += term
    println(typeof(result))
end
println(result)


Expression
Expression
Expression
2*a + (5/2)*a*b^2 + 3*b*c


In [23]:
chmb_sgn = []

for (i,u) in enumerate(samplevectors)
    sgn_u = []
    for v in V
        v_lift = [v[1], v[2], QQ(1)]
        s = dot(u,v_lift)
        push!(sgn_u, sign(s))
    end
    push!(chmb_sgn, (repr = u, sgn = sgn_u))
end
for (repr, sgn) in chmb_sgn
    println("Representing Vector u =", repr, ": sign = ", sgn)
end

LoadError: MethodError: no method matching iterate(::Cone{QQFieldElem})
The function `iterate` exists, but no method is defined for this combination of argument types.

[0mClosest candidates are:
[0m  iterate([91m::zzModRing[39m)
[0m[90m   @[39m [36mNemo[39m [90m~/.julia/packages/Nemo/xSaoQ/src/flint/[39m[90m[4mnmod.jl:493[24m[39m
[0m  iterate([91m::zzModRing[39m, [91m::UInt64[39m)
[0m[90m   @[39m [36mNemo[39m [90m~/.julia/packages/Nemo/xSaoQ/src/flint/[39m[90m[4mnmod.jl:495[24m[39m
[0m  iterate([91m::Compiler.MethodLookupResult[39m, Any...)
[0m[90m   @[39m [90mBase[39m [90m~/.julia/juliaup/julia-1.12.5+0.x64.linux.gnu/share/julia/Compiler/src/[39m[90m[4mmethodtable.jl:11[24m[39m
[0m  ...


In [ ]:
#PRE: list of vertices that span the polytope 
#POST: returns a vector that contains the equations 2p-Aq=0 where f=p/q is a local expression of the volume (i.e. the equations we need to solve to find the halving hyperplan), together with the corresponding chamber
function halving_equations(V)
    R, (a, b, c) = polynomial_ring(QQ, ["a", "b", "c"])
    S = fraction_field(R)
    equations = []
    
    for tuple in fcts
        ch = tuple.chamber
        fct = tuple.local_area
        p = numerator(fct)
        q = denominator(fct)
        f = 2*p - A*q
        push!(equations,(ch = ch, repr = tuple.repr, volume_ch = f ))
    end
    return equations
end

In [ ]:
R, (a, b, c) = polynomial_ring(QQ, ["a", "b", "c"])
equations = halving_equations(V)
relations = []
for tuple in equations
    local_vol = tuple.volume_ch
    chamber_ineq = facets(tuple.ch)
    push!(relations, (equalities = [local_vol], ineq = [chamber_ineq]))
end
println(relations)

In [ ]:
ch1 = equations[1].ch

In [ ]:
ch2 = equations[2].ch

In [ ]:
#PRE: Let V be a set of vertices
#POST: returns the representing vector u and its position with respect to the vertices, in other words, we calculate in which chamber u is contained. In altre parole pr ogni camera, calcolo le ineguaglianze che la descrivono, usando il vettore rappresentante
function chambers_conditions(V)
    chmb_sgn = []
    represent = representing_vectors(V)
    for (i,u) in enumerate(represent)
        sgn_u = []
        for v in V
            push!(sgn_u, sign(u[1]*v[1] + u[2] * v[2] + u[3]))
        end
        push!(chmb_sgn, (repr = u, sgn = sgn_u))
    end
    return chmb_sgn
end

In [ ]:
#PRE: Let V be a set of vertices
#POST: returns the representing vector u of each chamber together with the corresponding inequalities. 
function chambers_inequalities(V)
    R, (a, b, c) = polynomial_ring(QQ, ["a", "b", "c"])
    chmb_data = chambers_conditions(V)

    chmb_ineq = []
    for item in chmb_data
        sgn = item.sgn
        ineq_u = []
        for (i,v) in enumerate(V)
            s = inner_product(v,a,b,c)
            push!(ineq_u, sgn[i]*s)
        end
        push!(chmb_ineq, (repr = u, inequalities = ineq_u))
    end
    return chmb_ineq
end

In [ ]:
function halving_hyperplanes(V)
    R, (a, b, c) = polynomial_ring(QQ, ["a", "b", "c"])
    equations = halving_equations(V)
    inequalities = chambers_inequalities(V)

    relations = []
    for eq in equations
        found = false
        for ineq in inequalities
            if (eq.repr == ineq.repr)
                push!(relations, (chamber = eq.ch, repr = eq.repr, equality = eq.volume_ch, ineq = [ineq.inequalities]))
                found = true
                break
            end
        end
    end
    return relations
end


In [ ]:
println(halving_hyperplanes(V))